In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_GetFullLoadWorklist
# MAGIC Publishes global Full Load work across all eligible connection-owned
# MAGIC registrations. This notebook performs metadata reads only: no adapter, secret,
# MAGIC or JDBC access.

# COMMAND ----------

import json
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("catalog", "da_accelerators")
dbutils.widgets.text("control_schema", "control")
dbutils.widgets.text("max_tables", "0")
dbutils.widgets.text("only_connection_ids", "")
dbutils.widgets.text("only_source_table_ids", "")

run_id = dbutils.widgets.get("run_id").strip()
catalog = dbutils.widgets.get("catalog").strip()
control_schema = dbutils.widgets.get("control_schema").strip()
only_connection_ids = [
    value.strip() for value in
    dbutils.widgets.get("only_connection_ids").split(",") if value.strip()
]
only_source_table_ids = [
    value.strip() for value in
    dbutils.widgets.get("only_source_table_ids").split(",") if value.strip()
]

for name, value in (("run_id", run_id), ("catalog", catalog),
                    ("control_schema", control_schema)):
    if not value:
        raise ValueError(f"{name} is required")
try:
    max_tables = int(dbutils.widgets.get("max_tables").strip() or "0")
except ValueError as exc:
    raise ValueError("max_tables must be a non-negative integer") from exc
if max_tables < 0:
    raise ValueError("max_tables must be a non-negative integer")
if len(only_connection_ids) != len(set(only_connection_ids)):
    raise ValueError("only_connection_ids contains duplicate values")
if len(only_source_table_ids) != len(set(only_source_table_ids)):
    raise ValueError("only_source_table_ids contains duplicate values")

def _fqn(table_name):
    parts = (catalog, control_schema, table_name)
    return ".".join("`" + part.replace("`", "``") + "`" for part in parts)

control = spark.table(_fqn("source_table_control")).alias("c")
connections = spark.table(_fqn("source_connection")).alias("sc")

eligible = (
    control.join(
        connections,
        F.col("c.connection_id") == F.col("sc.connection_id"),
        "inner",
    )
    .filter(F.col("c.connection_id").isNotNull())
    .filter(F.trim(F.col("c.connection_id")) != "")
    .filter(F.col("c.source_table_id").isNotNull())
    .filter(F.trim(F.col("c.source_table_id")) != "")
    .filter(F.col("c.is_active") == F.lit(True))
    .filter(F.upper(F.col("c.table_decision")) == F.lit("AUTO_MIGRATE"))
    .filter(F.coalesce(F.col("c.initial_load_completed"), F.lit(False)) == F.lit(False))
    .filter(F.col("c.target_catalog").isNotNull() & (F.trim(F.col("c.target_catalog")) != ""))
    .filter(F.col("c.target_schema").isNotNull() & (F.trim(F.col("c.target_schema")) != ""))
    .filter(F.col("c.target_table").isNotNull() & (F.trim(F.col("c.target_table")) != ""))
    .filter(F.col("sc.is_active") == F.lit(True))
    .filter(F.upper(F.col("sc.connection_status")) == F.lit("VALID"))
    .filter(F.col("sc.secret_scope").isNotNull() & (F.trim(F.col("sc.secret_scope")) != ""))
    .filter(F.lower(F.trim(F.col("c.source_system"))) == F.lower(F.trim(F.col("sc.source_system"))))
)

if "source_identity_version" in control.columns:
    eligible = eligible.filter(F.col("c.source_identity_version") == F.lit(2))
if only_connection_ids:
    eligible = eligible.filter(F.col("c.connection_id").isin(only_connection_ids))
if only_source_table_ids:
    eligible = eligible.filter(F.col("c.source_table_id").isin(only_source_table_ids))

eligible = (
    eligible.select(
        F.lit(run_id).alias("run_id"),
        F.col("c.connection_id").alias("connection_id"),
        F.col("c.source_table_id").alias("source_table_id"),
        F.col("c.source_schema").alias("_source_schema"),
        F.col("c.source_table").alias("_source_table"),
    )
    .orderBy("connection_id", "_source_schema", "_source_table", "source_table_id")
)
if max_tables > 0:
    eligible = eligible.limit(max_tables)

rows = eligible.collect()
worklist = [
    {
        "run_id": row["run_id"],
        "connection_id": row["connection_id"],
        "source_table_id": row["source_table_id"],
    }
    for row in rows
]
keys = {(item["run_id"], item["connection_id"], item["source_table_id"])
        for item in worklist}
if len(keys) != len(worklist):
    raise ValueError("Full Load worklist contains duplicate ownership keys")

connection_count = len({item["connection_id"] for item in worklist})
payload_bytes = len(json.dumps(worklist, separators=(",", ":")).encode("utf-8"))
if payload_bytes > 40000:
    raise ValueError(
        f"Worklist payload is {payload_bytes} bytes and risks the task-value limit; "
        "reduce max_tables or use filters")

dbutils.jobs.taskValues.set(key="worklist", value=worklist)
dbutils.jobs.taskValues.set(key="worklist_count", value=len(worklist))
dbutils.jobs.taskValues.set(key="connection_count", value=connection_count)
print(f"Full Load worklist: tables={len(worklist)}, connections={connection_count}")

dbutils.notebook.exit(json.dumps({
    "status": "SUCCEEDED",
    "run_id": run_id,
    "worklist_count": len(worklist),
    "connection_count": connection_count,
    "worklist": worklist,
}))